# Universal Wearable Cardiac Triage System
**School Project — Tier 1 of a two-tier cardiac monitoring pipeline**

```
┌─────────────────────────────────────────────────────────────────┐
│                     TIER 1 — This Notebook                      │
│  Any Device (ECG + PPG) → Universal Adapter → Triage Model      │
│                                    ↓                            │
│              GREEN        YELLOW           RED                  │
│           All clear    User alert    Emergency escalation        │
└────────────────────────┬────────────────────────────────────────┘
                         │ RED or user-requested
┌────────────────────────▼────────────────────────────────────────┐
│                     TIER 2 — Clinical Notebook                  │
│         Full 12-lead CNN + BiLSTM → Cardiovascular outcomes     │
└─────────────────────────────────────────────────────────────────┘
```

**Supported devices:** Apple Watch · Samsung Galaxy Watch · Fitbit · Garmin · KardiaMobile · Smartphone camera  
**Detects:** AFib · Bradycardia · Tachycardia · Anomalies · SpO2 · Stress  
**Escalation:** Emergency contact + emergency services (GPS) simultaneously  
**Model size:** <500K parameters for real-time mobile inference

## 1 · Dependencies

In [ ]:
# !pip install torch numpy scipy matplotlib seaborn scikit-learn tqdm neurokit2

In [ ]:
import os, json, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import signal as scipy_signal
from scipy.interpolate import interp1d
from tqdm.notebook import tqdm
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple
from enum import Enum
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
torch.manual_seed(42)
np.random.seed(42)

## 2 · System Configuration

In [1]:
# ── Signal parameters ────────────────────────────────────────────────────────
ECG_FS       = 256          # Hz — typical wearable ECG sampling rate
PPG_FS       = 64           # Hz — typical wearable PPG sampling rate
WINDOW_SEC   = 30           # seconds of signal per inference window
ECG_LEN      = ECG_FS * WINDOW_SEC    # 7680 samples
PPG_LEN      = PPG_FS * WINDOW_SEC    # 1920 samples

# ── Detection targets ────────────────────────────────────────────────────────
RHYTHM_CLASSES = ['Normal', 'AFib', 'Bradycardia', 'Tachycardia', 'Anomaly']
N_RHYTHM       = len(RHYTHM_CLASSES)
STRESS_CLASSES = ['Low', 'Medium', 'High']

# ── Physiological thresholds ─────────────────────────────────────────────────
HR_BRADY_THRESH   = 50      # bpm — below this → Bradycardia alert
HR_TACHY_THRESH   = 120     # bpm — above this → Tachycardia alert
HR_CRITICAL_LOW   = 40      # bpm — immediate emergency
HR_CRITICAL_HIGH  = 150     # bpm — immediate emergency
SPO2_WARN_THRESH  = 95.0    # % — below this → warning
SPO2_CRIT_THRESH  = 90.0    # % — below this → emergency
HRV_LOW_THRESH    = 20.0    # ms RMSSD — below this → high stress

# ── Severity score thresholds (0–1 scale) ────────────────────────────────────
YELLOW_THRESH = 0.40        # above → YELLOW alert
RED_THRESH    = 0.72        # above → RED / emergency escalation

# ── Training ─────────────────────────────────────────────────────────────────
BATCH_SIZE   = 64
EPOCHS       = 40
LR           = 2e-3
WEIGHT_DECAY = 1e-4

print('Configuration loaded ✓')

Configuration loaded ✓


## 3 · Severity Tier System

In [ ]:
class Severity(Enum):
    GREEN  = 'GREEN'    # Normal — continue monitoring
    YELLOW = 'YELLOW'   # Warning — alert user, recommend clinical ECG
    RED    = 'RED'      # Critical — emergency escalation


@dataclass
class TriageResult:
    """
    Complete output from a single triage inference window.
    All downstream logic (alerts, escalation) is driven by this object.
    """
    timestamp:        str
    severity:         Severity
    severity_score:   float              # 0.0 (normal) → 1.0 (critical)

    # Rhythm
    rhythm_label:     str
    rhythm_probs:     Dict[str, float]   # per-class probabilities

    # Vitals
    heart_rate:       float              # bpm
    hrv_rmssd:        float              # ms
    spo2:             float              # %
    stress_level:     str               # Low / Medium / High
    stress_probs:     Dict[str, float]

    # Escalation
    requires_escalation: bool = False
    escalation_reason:   str  = ''
    gps_location:        Optional[Tuple[float, float]] = None

    def summary(self) -> str:
        colour = {'GREEN': '🟢', 'YELLOW': '🟡', 'RED': '🔴'}[self.severity.value]
        return (
            f"{colour} [{self.severity.value}] @ {self.timestamp}\n"
            f"  Rhythm : {self.rhythm_label} "
            f"(confidence {self.rhythm_probs[self.rhythm_label]:.1%})\n"
            f"  HR     : {self.heart_rate:.0f} bpm  "
            f"HRV: {self.hrv_rmssd:.1f} ms  "
            f"SpO2: {self.spo2:.1f}%\n"
            f"  Stress : {self.stress_level}  "
            f"Severity score: {self.severity_score:.3f}\n"
            + (f"  ⚠️  ESCALATION: {self.escalation_reason}" if self.requires_escalation else '')
        )


print('Severity tier system defined ✓')

## 4 · Signal Preprocessing

In [ ]:
class ECGPreprocessor:
    """
    Cleans raw single-lead ECG from wearable.
    Handles baseline wander, powerline noise, and motion artefacts.
    """
    def __init__(self, fs: int = ECG_FS):
        self.fs = fs
        # Bandpass 0.5–40 Hz removes baseline wander + high-freq noise
        self.sos = scipy_signal.butter(
            4, [0.5, 40.0], btype='bandpass', fs=fs, output='sos'
        )

    def __call__(self, ecg: np.ndarray) -> np.ndarray:
        ecg = np.nan_to_num(ecg.astype(np.float32))
        ecg = scipy_signal.sosfiltfilt(self.sos, ecg)   # zero-phase filter
        # Clip extreme artefacts (> 5 std)
        mu, std = ecg.mean(), ecg.std() + 1e-8
        ecg = np.clip(ecg, mu - 5*std, mu + 5*std)
        # Z-score normalise
        return (ecg - mu) / std


class PPGPreprocessor:
    """
    Cleans raw PPG signal from optical wearable sensor.
    Removes DC offset and motion artefacts via bandpass filter.
    """
    def __init__(self, fs: int = PPG_FS):
        self.fs = fs
        # Bandpass 0.5–8 Hz — keeps cardiac pulse, removes motion/breathing
        self.sos = scipy_signal.butter(
            4, [0.5, 8.0], btype='bandpass', fs=fs, output='sos'
        )

    def __call__(self, ppg: np.ndarray) -> np.ndarray:
        ppg = np.nan_to_num(ppg.astype(np.float32))
        ppg = scipy_signal.sosfiltfilt(self.sos, ppg)
        mu, std = ppg.mean(), ppg.std() + 1e-8
        return (ppg - mu) / std


class FeatureExtractor:
    """
    Extracts physiological features from preprocessed signals.
    Used for rule-based validation alongside model predictions.
    """
    def __init__(self, ecg_fs: int = ECG_FS, ppg_fs: int = PPG_FS):
        self.ecg_fs = ecg_fs
        self.ppg_fs = ppg_fs

    def extract_hr_from_ppg(self, ppg: np.ndarray) -> float:
        """Estimate heart rate from PPG peak detection."""
        peaks, _ = scipy_signal.find_peaks(
            ppg, distance=int(self.ppg_fs * 0.4),   # min 40bpm
            height=0.3
        )
        if len(peaks) < 2:
            return 75.0   # fallback
        rr_intervals = np.diff(peaks) / self.ppg_fs   # seconds
        return float(60.0 / np.median(rr_intervals))

    def extract_hrv_rmssd(self, ppg: np.ndarray) -> float:
        """RMSSD from successive RR interval differences — HRV proxy for stress."""
        peaks, _ = scipy_signal.find_peaks(
            ppg, distance=int(self.ppg_fs * 0.4), height=0.3
        )
        if len(peaks) < 3:
            return 30.0   # fallback — normal HRV
        rr_ms = np.diff(peaks) / self.ppg_fs * 1000   # ms
        successive_diffs = np.diff(rr_ms)
        return float(np.sqrt(np.mean(successive_diffs**2)))

    def estimate_spo2(self, ppg: np.ndarray) -> float:
        """
        Simplified SpO2 estimation from single-channel PPG.
        Real devices use red + infrared channels; this approximates
        from AC/DC ratio of the PPG waveform.
        In production, replace with actual red/IR channel ratio.
        """
        ac = ppg.std()
        dc = np.abs(ppg.mean()) + 1e-8
        r  = ac / dc
        # Empirical calibration curve: SpO2 ≈ 110 - 25*R
        spo2 = float(np.clip(110.0 - 25.0 * r, 85.0, 100.0))
        return spo2

    def extract_all(self, ecg: np.ndarray, ppg: np.ndarray) -> dict:
        hr    = self.extract_hr_from_ppg(ppg)
        hrv   = self.extract_hrv_rmssd(ppg)
        spo2  = self.estimate_spo2(ppg)
        return {'heart_rate': hr, 'hrv_rmssd': hrv, 'spo2': spo2}


ecg_prep = ECGPreprocessor()
ppg_prep = PPGPreprocessor()
feat_ext = FeatureExtractor()
print('Preprocessors ready ✓')

## 5 · Universal Device Input Adapter

In [ ]:
from scipy.signal import resample_poly
from math import gcd

# ── Device profiles — add new devices here ───────────────────────────────────
DEVICE_PROFILES = {
    # Device name          ECG Hz   PPG Hz   ECG available  PPG available
    'apple_watch':        {'ecg_fs': 512,  'ppg_fs': 100, 'has_ecg': True,  'has_ppg': True},
    'samsung_galaxy':     {'ecg_fs': 200,  'ppg_fs': 100, 'has_ecg': True,  'has_ppg': True},
    'fitbit_sense':       {'ecg_fs': 256,  'ppg_fs': 128, 'has_ecg': True,  'has_ppg': True},
    'fitbit_charge':      {'ecg_fs': None, 'ppg_fs': 128, 'has_ecg': False, 'has_ppg': True},
    'garmin_venu':        {'ecg_fs': 256,  'ppg_fs': 25,  'has_ecg': True,  'has_ppg': True},
    'kardia_mobile':      {'ecg_fs': 300,  'ppg_fs': None,'has_ecg': True,  'has_ppg': False},
    'kardia_mobile_6l':   {'ecg_fs': 300,  'ppg_fs': None,'has_ecg': True,  'has_ppg': False},
    'smartphone_camera':  {'ecg_fs': None, 'ppg_fs': 30,  'has_ecg': False, 'has_ppg': True},
    'generic_wearable':   {'ecg_fs': 256,  'ppg_fs': 64,  'has_ecg': True,  'has_ppg': True},
}


class UniversalInputAdapter:
    """
    Normalises raw signals from ANY supported device into the
    fixed format the triage model expects:
        ECG → 256 Hz, 7680 samples (30s)
        PPG →  64 Hz, 1920 samples (30s)

    Handles:
      ✓ Different sampling rates   — rational resampling (no quality loss)
      ✓ Different window lengths   — pad with zeros or crop from centre
      ✓ Missing ECG (phone/Fitbit) — zero tensor + modality mask flag
      ✓ Missing PPG (KardiaMobile) — zero tensor + modality mask flag
      ✓ Signal quality check       — rejects heavily clipped/flat signals
    """
    TARGET_ECG_FS  = ECG_FS      # 256 Hz
    TARGET_PPG_FS  = PPG_FS      # 64 Hz
    TARGET_ECG_LEN = ECG_LEN     # 7680
    TARGET_PPG_LEN = PPG_LEN     # 1920

    def __init__(self, device_name: str = 'generic_wearable'):
        if device_name not in DEVICE_PROFILES:
            raise ValueError(
                f'Unknown device: {device_name}\n'
                f'Supported: {list(DEVICE_PROFILES.keys())}'
            )
        self.profile     = DEVICE_PROFILES[device_name]
        self.device_name = device_name
        print(f'Adapter configured for: {device_name}')
        print(f'  ECG: {"✓ " + str(self.profile["ecg_fs"]) + " Hz" if self.profile["has_ecg"] else "✗ not available"}')
        print(f'  PPG: {"✓ " + str(self.profile["ppg_fs"]) + " Hz" if self.profile["has_ppg"] else "✗ not available"}')

    def _rational_resample(self, sig: np.ndarray,
                           src_fs: int, tgt_fs: int) -> np.ndarray:
        """Lossless rational resampling using GCD reduction."""
        if src_fs == tgt_fs:
            return sig
        g   = gcd(src_fs, tgt_fs)
        up  = tgt_fs // g
        down = src_fs // g
        return resample_poly(sig, up, down).astype(np.float32)

    def _fit_length(self, sig: np.ndarray, target_len: int) -> np.ndarray:
        """Crops from centre if too long, zero-pads symmetrically if too short."""
        L = len(sig)
        if L == target_len:
            return sig
        if L > target_len:
            # Centre crop
            start = (L - target_len) // 2
            return sig[start:start + target_len]
        # Symmetric zero padding
        pad_total = target_len - L
        pad_left  = pad_total // 2
        pad_right = pad_total - pad_left
        return np.pad(sig, (pad_left, pad_right), mode='constant')

    def _check_signal_quality(self, sig: np.ndarray,
                               name: str) -> Tuple[bool, str]:
        """Rejects flat lines, heavily clipped, or all-NaN signals."""
        if np.all(np.isnan(sig)):
            return False, f'{name}: all NaN — sensor disconnected?'
        if sig.std() < 1e-6:
            return False, f'{name}: flat line — lead off or no contact'
        clipped = np.mean(np.abs(sig) > 4.5 * sig.std())
        if clipped > 0.15:
            return False, f'{name}: {clipped:.0%} samples clipped — motion artefact'
        return True, 'ok'

    def adapt(self,
              ecg_raw: Optional[np.ndarray] = None,
              ppg_raw: Optional[np.ndarray] = None
              ) -> Dict[str, object]:
        """
        Main entry point.

        Returns dict with:
          'ecg'          : np.ndarray (TARGET_ECG_LEN,)  — ready for model
          'ppg'          : np.ndarray (TARGET_PPG_LEN,)  — ready for model
          'has_ecg'      : bool
          'has_ppg'      : bool
          'quality_flags': dict of signal quality warnings
          'device'       : str
        """
        result       = {'device': self.device_name, 'quality_flags': {}}
        has_ecg      = self.profile['has_ecg'] and ecg_raw is not None
        has_ppg      = self.profile['has_ppg'] and ppg_raw is not None

        # ── ECG processing ────────────────────────────────────────────────
        if has_ecg:
            ok, msg = self._check_signal_quality(ecg_raw, 'ECG')
            result['quality_flags']['ecg'] = msg
            if ok:
                ecg = self._rational_resample(
                    ecg_raw, self.profile['ecg_fs'], self.TARGET_ECG_FS
                )
                ecg = self._fit_length(ecg, self.TARGET_ECG_LEN)
                ecg = ecg_prep(ecg)
            else:
                print(f'⚠️  ECG quality warning: {msg} — using zeros')
                ecg    = np.zeros(self.TARGET_ECG_LEN, dtype=np.float32)
                has_ecg = False
        else:
            ecg = np.zeros(self.TARGET_ECG_LEN, dtype=np.float32)
            result['quality_flags']['ecg'] = 'not available on this device'

        # ── PPG processing ────────────────────────────────────────────────
        if has_ppg:
            ok, msg = self._check_signal_quality(ppg_raw, 'PPG')
            result['quality_flags']['ppg'] = msg
            if ok:
                ppg = self._rational_resample(
                    ppg_raw, self.profile['ppg_fs'], self.TARGET_PPG_FS
                )
                ppg = self._fit_length(ppg, self.TARGET_PPG_LEN)
                ppg = ppg_prep(ppg)
            else:
                print(f'⚠️  PPG quality warning: {msg} — using zeros')
                ppg    = np.zeros(self.TARGET_PPG_LEN, dtype=np.float32)
                has_ppg = False
        else:
            ppg = np.zeros(self.TARGET_PPG_LEN, dtype=np.float32)
            result['quality_flags']['ppg'] = 'not available on this device'

        # ── Modality availability flags ───────────────────────────────────
        result.update({
            'ecg':     ecg,
            'ppg':     ppg,
            'has_ecg': has_ecg,
            'has_ppg': has_ppg,
        })

        if not has_ecg and not has_ppg:
            raise RuntimeError('No valid signal available — cannot run triage')

        return result


# ── Demo: test adapter across all device types ───────────────────────────────
print('\n--- Adapter compatibility test ---')
for device_name, profile in DEVICE_PROFILES.items():
    adapter = UniversalInputAdapter(device_name)
    # Simulate raw signals at device-native sampling rate
    ecg_raw = (np.random.randn(profile['ecg_fs'] * 30).astype(np.float32)
               if profile['has_ecg'] else None)
    ppg_raw = (np.random.randn(profile['ppg_fs'] * 30).astype(np.float32)
               if profile['has_ppg'] else None)
    out = adapter.adapt(ecg_raw, ppg_raw)
    print(f'  → ECG: {out["ecg"].shape}  PPG: {out["ppg"].shape}  '
          f'has_ecg={out["has_ecg"]}  has_ppg={out["has_ppg"]}\n')

## 6 · Confidence Scaling by Device Tier

In [ ]:
"""
When only one modality is available, model confidence is reduced.
The severity threshold for RED escalation is tightened accordingly
to compensate for missing signal — better to escalate cautiously
than to miss a critical event.
"""

CONFIDENCE_PROFILES = {
    # Modality combo       confidence  adjusted RED thresh  note
    'ecg_and_ppg':        (1.00,       RED_THRESH,         'Full capability'),
    'ecg_only':           (0.80,       RED_THRESH - 0.08,  'No PPG — SpO2/stress limited'),
    'ppg_only':           (0.65,       RED_THRESH - 0.15,  'No ECG — morphology limited'),
}

def get_confidence_profile(has_ecg: bool, has_ppg: bool) -> dict:
    key = ('ecg_and_ppg' if has_ecg and has_ppg
           else 'ecg_only' if has_ecg
           else 'ppg_only')
    conf, thresh, note = CONFIDENCE_PROFILES[key]
    return {'confidence': conf, 'red_threshold': thresh,
            'note': note, 'modality': key}

# Preview confidence profiles
print('Confidence profiles by available modality:\n')
for key, (conf, thresh, note) in CONFIDENCE_PROFILES.items():
    print(f'  {key:<15}  confidence={conf:.0%}  RED@{thresh:.2f}  ({note})')

## 7 · Updated Severity Engine (Device-Aware)

In [ ]:
class UniversalSeverityEngine:
    """
    Device-aware severity engine.
    Wraps the original SeverityEngine but adjusts thresholds
    based on which modalities are available from the device.
    """
    def __init__(self, model: nn.Module, device: str = DEVICE):
        self.model  = model
        self.device = device
        self.model.eval()

    @torch.no_grad()
    def evaluate(self,
                 adapter_output: dict,
                 gps: Optional[Tuple[float, float]] = None) -> TriageResult:

        ecg     = adapter_output['ecg']
        ppg     = adapter_output['ppg']
        has_ecg = adapter_output['has_ecg']
        has_ppg = adapter_output['has_ppg']

        # Get adjusted thresholds for this device's modality combo
        conf_profile = get_confidence_profile(has_ecg, has_ppg)
        red_thresh   = conf_profile['red_threshold']

        # Extract rule-based vitals (only from PPG if available)
        if has_ppg:
            vitals = feat_ext.extract_all(ecg, ppg)
        else:
            # ECG-only: estimate HR from RR peaks in ECG
            peaks, _ = scipy_signal.find_peaks(ecg, distance=int(ECG_FS * 0.4))
            hr  = float(60.0 / np.median(np.diff(peaks) / ECG_FS)) if len(peaks) > 2 else 75.0
            vitals = {'heart_rate': hr, 'hrv_rmssd': 30.0, 'spo2': 97.0}

        # Model inference
        ecg_t = torch.FloatTensor(ecg).unsqueeze(0).unsqueeze(0).to(self.device)
        ppg_t = torch.FloatTensor(ppg).unsqueeze(0).unsqueeze(0).to(self.device)
        outputs = self.model(ecg_t, ppg_t)

        rhythm_probs   = F.softmax(outputs['rhythm_logits'], dim=1).cpu().numpy()[0]
        stress_probs   = F.softmax(outputs['stress_logits'], dim=1).cpu().numpy()[0]
        spo2_pred      = outputs['spo2_pred'].cpu().item()
        severity_score = outputs['severity_score'].cpu().item()

        # Scale severity score by modality confidence
        # Lower confidence = inflate score slightly toward caution
        adjusted_score = severity_score / conf_profile['confidence']
        adjusted_score = float(np.clip(adjusted_score, 0.0, 1.0))

        rhythm_label = RHYTHM_CLASSES[int(rhythm_probs.argmax())]
        stress_label = STRESS_CLASSES[int(stress_probs.argmax())]

        if has_ppg:
            blended_spo2 = 0.6 * spo2_pred + 0.4 * vitals['spo2']
        else:
            blended_spo2 = vitals['spo2']   # fallback only
        vitals['spo2'] = blended_spo2

        # Severity classification with adjusted threshold
        severity, reason = self._classify_severity(
            adjusted_score, vitals, rhythm_label, red_thresh
        )

        return TriageResult(
            timestamp           = datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            severity            = severity,
            severity_score      = adjusted_score,
            rhythm_label        = rhythm_label,
            rhythm_probs        = {c: float(f'{p:.4f}') for c, p in zip(RHYTHM_CLASSES, rhythm_probs)},
            heart_rate          = vitals['heart_rate'],
            hrv_rmssd           = vitals['hrv_rmssd'],
            spo2                = blended_spo2,
            stress_level        = stress_label,
            stress_probs        = {c: float(f'{p:.4f}') for c, p in zip(STRESS_CLASSES, stress_probs)},
            requires_escalation = (severity == Severity.RED),
            escalation_reason   = reason,
            gps_location        = gps
        )

    def _classify_severity(self, score, vitals, rhythm, red_thresh):
        # Hard physiological overrides
        if vitals['heart_rate'] < HR_CRITICAL_LOW:
            return Severity.RED, f'Critical bradycardia: {vitals["heart_rate"]:.0f} bpm'
        if vitals['heart_rate'] > HR_CRITICAL_HIGH:
            return Severity.RED, f'Critical tachycardia: {vitals["heart_rate"]:.0f} bpm'
        if vitals['spo2'] < SPO2_CRIT_THRESH:
            return Severity.RED, f'Critical hypoxia: SpO2 {vitals["spo2"]:.1f}%'
        if rhythm == 'AFib' and score > 0.6:
            return Severity.RED, 'High-confidence AFib detected'
        if score >= red_thresh:
            return Severity.RED, f'Severity score {score:.2f} exceeds threshold {red_thresh:.2f}'
        if score >= YELLOW_THRESH:
            return Severity.YELLOW, f'{rhythm} · Score {score:.2f}'
        return Severity.GREEN, ''


universal_engine = UniversalSeverityEngine(triage_model)
print('Universal severity engine ready ✓')

## 8 · Universal End-to-End Demo — All Device Types

In [ ]:
user = UserProfile(
    name              = 'John Doe',
    age               = 45,
    blood_type        = 'O+',
    known_conditions  = ['Hypertension'],
    medications       = ['Lisinopril 10mg'],
    emergency_contact = {'name': 'Jane Doe', 'phone': '+1-555-0100', 'relation': 'Spouse'},
    emergency_number  = '911'
)
escalation_system = EscalationSystem(user)

# Test every device type with an AFib signal
print('\n=== Universal Device Compatibility Test (AFib scenario) ===\n')
for device_name, profile in DEVICE_PROFILES.items():
    adapter = UniversalInputAdapter(device_name)
    ecg_raw = (generate_ecg('AFib', fs=profile['ecg_fs'])
               if profile['has_ecg'] else None)
    ppg_raw = (generate_ppg(110, 93, 'High', fs=profile['ppg_fs'])
               if profile['has_ppg'] else None)

    adapted = adapter.adapt(ecg_raw, ppg_raw)
    result  = universal_engine.evaluate(adapted, gps=(4.65, 7.45))

    print(f'[{device_name}]')
    print(result.summary())
    if result.requires_escalation:
        print('  → Escalation triggered')
    print()

## 9 · Signal Quality Visualiser

In [ ]:
# Visualise adapter resampling across 3 device types side by side
devices_to_plot = ['apple_watch', 'garmin_venu', 'smartphone_camera']
fig, axes = plt.subplots(len(devices_to_plot), 2, figsize=(16, 10))
t_ecg = np.arange(ECG_LEN) / ECG_FS
t_ppg = np.arange(PPG_LEN) / PPG_FS

for i, device_name in enumerate(devices_to_plot):
    profile = DEVICE_PROFILES[device_name]
    adapter = UniversalInputAdapter(device_name)
    ecg_raw = (generate_ecg('Normal', fs=profile['ecg_fs']) if profile['has_ecg'] else None)
    ppg_raw = (generate_ppg(72, 98, 'Low', fs=profile['ppg_fs']) if profile['has_ppg'] else None)
    adapted = adapter.adapt(ecg_raw, ppg_raw)

    # ECG
    ax = axes[i, 0]
    if adapted['has_ecg']:
        ax.plot(t_ecg, adapted['ecg'], color='#1f77b4', linewidth=0.7)
        ax.set_title(f'{device_name} — ECG ({profile["ecg_fs"]}Hz → {ECG_FS}Hz)',
                     fontsize=9, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'ECG not available\n(zeros passed to model)',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=10, color='grey')
        ax.set_title(f'{device_name} — ECG: N/A', fontsize=9)
    ax.set_ylabel('Amplitude'); ax.grid(alpha=0.3)

    # PPG
    ax = axes[i, 1]
    if adapted['has_ppg']:
        ax.plot(t_ppg, adapted['ppg'], color='#e15759', linewidth=0.7)
        ax.set_title(f'{device_name} — PPG ({profile["ppg_fs"]}Hz → {PPG_FS}Hz)',
                     fontsize=9, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'PPG not available\n(zeros passed to model)',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=10, color='grey')
        ax.set_title(f'{device_name} — PPG: N/A', fontsize=9)
    ax.set_ylabel('Amplitude'); ax.grid(alpha=0.3)

for ax in axes[-1]:
    ax.set_xlabel('Time (s)')

fig.suptitle('Universal Adapter Output — Signals Normalised to Model Format',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 10 · Training — Multi-Task Loss

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Lightweight building blocks — optimised for mobile inference
# ─────────────────────────────────────────────────────────────────────────────

class DepthwiseSeparableConv(nn.Module):
    """
    Depthwise separable convolution — same receptive field as regular Conv1D
    but 8–9× fewer parameters. Used in MobileNet; key to keeping <500K params.
    """
    def __init__(self, in_ch, out_ch, kernel_size=7, stride=1):
        super().__init__()
        pad = kernel_size // 2
        self.dw = nn.Conv1d(in_ch, in_ch, kernel_size, stride=stride,
                            padding=pad, groups=in_ch, bias=False)  # per-channel
        self.pw = nn.Conv1d(in_ch, out_ch, 1, bias=False)           # cross-channel
        self.bn = nn.BatchNorm1d(out_ch)

    def forward(self, x):
        return F.relu(self.bn(self.pw(self.dw(x))), inplace=True)


class LightResBlock(nn.Module):
    """Residual block using depthwise separable convolutions."""
    def __init__(self, channels, kernel_size=7, dropout=0.1):
        super().__init__()
        self.conv1 = DepthwiseSeparableConv(channels, channels, kernel_size)
        self.conv2 = DepthwiseSeparableConv(channels, channels, kernel_size)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        return x + self.drop(self.conv2(self.conv1(x)))


class SignalEncoder(nn.Module):
    """
    Shared encoder branch for ECG or PPG.
    Progressively compresses the signal into a fixed-size feature vector.
    """
    def __init__(self, in_len: int, out_dim: int = 128, dropout: float = 0.1):
        super().__init__()
        self.encoder = nn.Sequential(
            # Entry: capture local waveform shape
            nn.Conv1d(1, 16, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(4),                    # /4

            # Stage 1
            DepthwiseSeparableConv(16, 32, stride=2),   # /2
            LightResBlock(32, dropout=dropout),

            # Stage 2
            DepthwiseSeparableConv(32, 64, stride=2),   # /2
            LightResBlock(64, dropout=dropout),

            # Stage 3
            DepthwiseSeparableConv(64, out_dim, stride=2),  # /2
            LightResBlock(out_dim, dropout=dropout),

            nn.AdaptiveAvgPool1d(16)            # fixed temporal output: 16 steps
        )

    def forward(self, x):   # x: (B, 1, T)
        return self.encoder(x)  # → (B, out_dim, 16)


# ─────────────────────────────────────────────────────────────────────────────
# Full triage model
# ─────────────────────────────────────────────────────────────────────────────

class WearableTriageModel(nn.Module):
    """
    Dual-branch ECG + PPG triage model.

    Inputs
    ------
    ecg : (B, 1, ECG_LEN)   — single-lead ECG at 256 Hz
    ppg : (B, 1, PPG_LEN)   — PPG signal at 64 Hz

    Outputs (dict)
    -------
    rhythm_logits  : (B, 5)  — Normal/AFib/Brady/Tachy/Anomaly
    stress_logits  : (B, 3)  — Low/Medium/High stress
    spo2_pred      : (B, 1)  — SpO2 estimate (%)
    severity_score : (B, 1)  — 0–1 continuous severity

    Architecture
    ------------
    ECG encoder (SignalEncoder) ─────────────────────┐
                                                      ├─ cat → (B, 256, 16)
    PPG encoder (SignalEncoder) ─────────────────────┘
                                                      ↓
                                  Fusion Conv + BiLSTM (hidden=64, 1 layer)
                                                      ↓
                                       cat(fwd, bwd) → (B, 128)
                                                      ↓
                              ┌───────────┬───────────┬───────────┐
                         Rhythm       Stress       SpO2      Severity
                          head         head        head        head
    """
    def __init__(self, dropout: float = 0.2):
        super().__init__()
        FEAT_DIM = 128

        # ── Per-signal encoders ────────────────────────────────────────────
        self.ecg_encoder = SignalEncoder(ECG_LEN, out_dim=FEAT_DIM, dropout=dropout)
        self.ppg_encoder = SignalEncoder(PPG_LEN, out_dim=FEAT_DIM, dropout=dropout)

        # ── Fusion layer — merges ECG + PPG feature maps ──────────────────
        # Input: cat(ecg_feat, ppg_feat) = (B, 256, 16)
        self.fusion = nn.Sequential(
            nn.Conv1d(FEAT_DIM * 2, FEAT_DIM, kernel_size=1, bias=False),
            nn.BatchNorm1d(FEAT_DIM),
            nn.ReLU(inplace=True)
        )

        # ── BiLSTM temporal reasoning over fused features ─────────────────
        # 1 layer — lighter than the clinical model (16 timesteps, not 32)
        self.bilstm = nn.LSTM(
            input_size    = FEAT_DIM,
            hidden_size   = 64,
            num_layers    = 1,
            batch_first   = True,
            bidirectional = True
        )
        # BiLSTM output: 64*2 = 128 dims
        self.lstm_drop = nn.Dropout(dropout)

        # ── Multi-task prediction heads ────────────────────────────────────
        def make_head(out_dim):
            return nn.Sequential(
                nn.Linear(128, 64),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
                nn.Linear(64, out_dim)
            )

        self.rhythm_head   = make_head(N_RHYTHM)  # 5 rhythm classes
        self.stress_head   = make_head(3)          # 3 stress levels
        self.spo2_head     = make_head(1)          # SpO2 regression
        self.severity_head = make_head(1)          # severity score 0–1

    def forward(self, ecg, ppg):
        # ── Encode each signal independently ──────────────────────────────
        ecg_feat = self.ecg_encoder(ecg)      # (B, 128, 16)
        ppg_feat = self.ppg_encoder(ppg)      # (B, 128, 16)

        # ── Fuse: concatenate along channel dim, then compress ─────────────
        fused = torch.cat([ecg_feat, ppg_feat], dim=1)  # (B, 256, 16)
        fused = self.fusion(fused)                       # (B, 128, 16)

        # ── BiLSTM: (B, C, T) → (B, T, C) → LSTM → context ───────────────
        fused = fused.permute(0, 2, 1)                  # (B, 16, 128)
        _, (hn, _) = self.bilstm(fused)
        context = torch.cat([hn[0], hn[1]], dim=1)      # (B, 128)
        context = self.lstm_drop(context)

        # ── Multi-task outputs ─────────────────────────────────────────────
        return {
            'rhythm_logits':  self.rhythm_head(context),          # (B, 5)
            'stress_logits':  self.stress_head(context),          # (B, 3)
            'spo2_pred':      torch.sigmoid(self.spo2_head(context)) * 15 + 85,  # 85–100%
            'severity_score': torch.sigmoid(self.severity_head(context)),  # 0–1
        }


triage_model = WearableTriageModel().to(DEVICE)
n_params = sum(p.numel() for p in triage_model.parameters() if p.requires_grad)
print(f'Triage model parameters: {n_params:,}   target: <500,000')
assert n_params < 500_000, f'Model too large: {n_params:,} params'

# Verify forward pass
with torch.no_grad():
    dummy_ecg = torch.randn(2, 1, ECG_LEN).to(DEVICE)
    dummy_ppg = torch.randn(2, 1, PPG_LEN).to(DEVICE)
    out = triage_model(dummy_ecg, dummy_ppg)
    print('Output shapes:')
    for k, v in out.items():
        print(f'  {k}: {tuple(v.shape)}')

## 11 · Real-World Dataset Integration (PTB-XL)

This project folder already contains PTB-XL, which is a real clinical ECG dataset. PTB-XL does **not** include wearable PPG, SpO2, or stress labels, so this section uses real ECG for rhythm training and fills the missing wearable channels with neutral placeholders. That keeps the model input shape consistent while making it clear which targets are real and which are synthetic/proxy.

Recommended real-world sources by task:

| Task | Dataset | Use in this notebook |
|---|---|---|
| ECG rhythm / abnormality | PTB-XL, MIT-BIH Arrhythmia | Train/evaluate ECG branch and rhythm head |
| PPG waveform / heart rate | MIMIC waveform, BIDMC PPG | Add real PPG branch training |
| Stress / affect | WESAD | Train stress head with wearable-like signals |
| SpO2 | MIMIC waveform/vitals | Replace placeholder SpO2 labels |

For your current folder, start with PTB-XL below.


In [2]:
from pathlib import Path
import ast

REAL_DATA_ROOT = Path.cwd()
PTBXL_DATABASE = REAL_DATA_ROOT / 'ptbxl_database.csv'
SCP_STATEMENTS = REAL_DATA_ROOT / 'scp_statements.csv'

try:
    import wfdb
    WFDB_AVAILABLE = True
except ImportError:
    WFDB_AVAILABLE = False
    print('wfdb is not installed. Run: pip install wfdb')


def load_ptbxl_metadata(root: Path = REAL_DATA_ROOT) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load PTB-XL metadata and expand SCP diagnostic codes."""
    database_path = root / 'ptbxl_database.csv'
    scp_path = root / 'scp_statements.csv'

    if not database_path.exists() or not scp_path.exists():
        raise FileNotFoundError('PTB-XL metadata files were not found in this folder.')

    meta = pd.read_csv(database_path, index_col='ecg_id')
    scp = pd.read_csv(scp_path, index_col=0)
    meta['scp_codes'] = meta['scp_codes'].apply(ast.literal_eval)
    return meta, scp


def ptbxl_primary_diagnostic(scp_codes: dict, scp: pd.DataFrame) -> str:
    """Return the highest-likelihood diagnostic superclass from SCP labels."""
    best_label, best_weight = 'NORM', -1.0
    for code, weight in scp_codes.items():
        if code in scp.index and bool(scp.loc[code].get('diagnostic', False)):
            superclass = scp.loc[code].get('diagnostic_class', 'NORM')
            if pd.notna(superclass) and weight > best_weight:
                best_label, best_weight = superclass, weight
    return str(best_label)


def map_ptbxl_to_triage_label(row: pd.Series, scp: pd.DataFrame) -> str:
    """
    Map PTB-XL diagnostic labels into this notebook's rhythm classes.

    PTB-XL is mostly diagnostic morphology/superclass labeling, not direct
    wearable rhythm labeling. This mapping is intentionally conservative:
    normal stays Normal; abnormal diagnostic classes become Anomaly unless
    the SCP code clearly implies AFib, bradycardia, or tachycardia.
    """
    codes = row['scp_codes']
    code_names = {str(k).upper() for k in codes.keys()}

    afib_codes = {'AFIB', 'AFLT'}
    brady_codes = {'SBRAD', 'BRAD'}
    tachy_codes = {'STACH', 'SVTAC', 'VTAC'}

    if code_names & afib_codes:
        return 'AFib'
    if code_names & brady_codes:
        return 'Bradycardia'
    if code_names & tachy_codes:
        return 'Tachycardia'

    superclass = ptbxl_primary_diagnostic(codes, scp)
    if superclass == 'NORM':
        return 'Normal'
    return 'Anomaly'


def fit_signal_length(sig: np.ndarray, target_len: int) -> np.ndarray:
    """Centre-crop or zero-pad a 1D signal to the model input length."""
    sig = np.asarray(sig, dtype=np.float32)
    if len(sig) == target_len:
        return sig
    if len(sig) > target_len:
        start = (len(sig) - target_len) // 2
        return sig[start:start + target_len]
    pad = target_len - len(sig)
    return np.pad(sig, (pad // 2, pad - pad // 2), mode='constant')


class PTBXLWearableDataset(Dataset):
    """
    PTB-XL adapter for this wearable triage model.

    Real labels:
      - ECG waveform
      - rhythm_label, mapped from PTB-XL SCP statements

    Placeholder/proxy labels:
      - PPG is zeros because PTB-XL has ECG only
      - SpO2 defaults to 98%
      - stress_label defaults to Low
      - severity is derived from the mapped rhythm class
    """
    def __init__(self,
                 root: Path = REAL_DATA_ROOT,
                 sampling_rate: int = 100,
                 max_records: Optional[int] = None,
                 lead: str = 'II'):
        if not WFDB_AVAILABLE:
            raise ImportError('wfdb is required to read PTB-XL waveform files. Run: pip install wfdb')

        self.root = Path(root)
        self.sampling_rate = sampling_rate
        self.lead = lead
        self.meta, self.scp = load_ptbxl_metadata(self.root)
        self.meta = self.meta.copy()
        self.meta['triage_label'] = self.meta.apply(
            lambda row: map_ptbxl_to_triage_label(row, self.scp), axis=1
        )

        if max_records is not None:
            self.meta = self.meta.head(max_records)

        self.records = self.meta.reset_index()
        self.ecg_prep = ECGPreprocessor(fs=ECG_FS)

    def __len__(self):
        return len(self.records)

    def _record_path(self, row: pd.Series) -> Path:
        column = 'filename_lr' if self.sampling_rate == 100 else 'filename_hr'
        return self.root / row[column]

    def _load_ecg(self, row: pd.Series) -> np.ndarray:
        record_path = self._record_path(row)
        signal, fields = wfdb.rdsamp(str(record_path))
        sig_names = fields.get('sig_name', [])
        lead_idx = sig_names.index(self.lead) if self.lead in sig_names else 0
        ecg = signal[:, lead_idx].astype(np.float32)

        # PTB-XL records are 10 seconds. Resample to the model's ECG rate,
        # then pad to the 30-second wearable inference window.
        src_fs = int(fields.get('fs', self.sampling_rate))
        if src_fs != ECG_FS:
            g = gcd(src_fs, ECG_FS)
            ecg = resample_poly(ecg, ECG_FS // g, src_fs // g).astype(np.float32)
        ecg = fit_signal_length(ecg, ECG_LEN)
        return self.ecg_prep(ecg).astype(np.float32)

    @staticmethod
    def _severity_from_label(label: str) -> float:
        return float({
            'Normal': 0.05,
            'AFib': 0.65,
            'Bradycardia': 0.50,
            'Tachycardia': 0.50,
            'Anomaly': 0.55,
        }[label])

    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        label = row['triage_label']
        ecg = self._load_ecg(row)
        ppg = np.zeros(PPG_LEN, dtype=np.float32)
        spo2 = 98.0
        stress_label = 0  # Low; PTB-XL has no stress labels
        severity = self._severity_from_label(label)

        return {
            'ecg': torch.FloatTensor(ecg).unsqueeze(0),
            'ppg': torch.FloatTensor(ppg).unsqueeze(0),
            'rhythm_label': torch.tensor(RHYTHM_CLASSES.index(label), dtype=torch.long),
            'stress_label': torch.tensor(stress_label, dtype=torch.long),
            'spo2': torch.FloatTensor([spo2]),
            'severity': torch.FloatTensor([severity]),
        }


if PTBXL_DATABASE.exists() and SCP_STATEMENTS.exists():
    meta_preview, scp_preview = load_ptbxl_metadata(REAL_DATA_ROOT)
    mapped_preview = meta_preview.head(1000).apply(
        lambda row: map_ptbxl_to_triage_label(row, scp_preview), axis=1
    )
    print(f'PTB-XL found: {len(meta_preview):,} records')
    print('Preview mapped class counts:')
    print(mapped_preview.value_counts())
    print('\nTo train on a small real ECG subset:')
    print('  real_ds = PTBXLWearableDataset(max_records=1000, sampling_rate=100)')
    print('  real_loader = DataLoader(real_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)')
else:
    print('PTB-XL metadata files were not found in this folder.')


NameError: name 'Tuple' is not defined

## 12 · Synthetic Dataset (for training without wearable data)

In [ ]:
"""
Generates physiologically plausible synthetic ECG + PPG signals per class.
Replace with real wearable dataset when available (e.g. MIT-BIH, MIMIC-III PPG,
or your own collected data via Apple HealthKit / Wear OS APIs).
"""

def generate_ecg(rhythm: str, fs: int = ECG_FS, duration: int = WINDOW_SEC) -> np.ndarray:
    """Generates synthetic single-lead ECG for a given rhythm class."""
    T = fs * duration
    t = np.linspace(0, duration, T)
    noise = np.random.normal(0, 0.05, T)

    if rhythm == 'Normal':
        hr = np.random.uniform(60, 100)
        rr = fs * 60 / hr
        beats = np.arange(0, T, rr).astype(int)
        ecg = noise.copy()
        for b in beats:
            if b + 30 < T:
                # P wave
                ecg[b:b+10]    += 0.15 * scipy_signal.windows.gaussian(10, 3)
                # QRS complex
                ecg[b+10:b+20] += 1.0  * scipy_signal.windows.gaussian(10, 2)
                # T wave
                ecg[b+20:b+30] += 0.25 * scipy_signal.windows.gaussian(10, 4)

    elif rhythm == 'AFib':
        # Irregular RR intervals + no P wave
        ecg = noise.copy()
        ecg += 0.08 * np.random.randn(T)   # fibrillatory baseline
        beat_pos = 0
        while beat_pos < T:
            rr = int(np.random.uniform(0.4, 1.2) * fs)
            if beat_pos + 20 < T:
                ecg[beat_pos:beat_pos+15] += 0.9 * scipy_signal.windows.gaussian(15, 2)
            beat_pos += rr

    elif rhythm == 'Bradycardia':
        hr = np.random.uniform(HR_CRITICAL_LOW, HR_BRADY_THRESH)
        rr = int(fs * 60 / hr)
        ecg = noise.copy()
        for b in range(0, T, rr):
            if b + 30 < T:
                ecg[b+10:b+20] += 1.0 * scipy_signal.windows.gaussian(10, 2)
                ecg[b+20:b+30] += 0.2 * scipy_signal.windows.gaussian(10, 4)

    elif rhythm == 'Tachycardia':
        hr = np.random.uniform(HR_TACHY_THRESH, HR_CRITICAL_HIGH)
        rr = int(fs * 60 / hr)
        ecg = noise.copy()
        for b in range(0, T, rr):
            if b + 15 < T:
                ecg[b:b+10] += 0.8 * scipy_signal.windows.gaussian(10, 2)

    else:  # Anomaly
        ecg = noise + 0.3 * np.sin(2 * np.pi * 1.5 * t) + np.random.randn(T) * 0.2

    return ecg_prep(ecg)


def generate_ppg(hr: float, spo2: float, stress: str,
                 fs: int = PPG_FS, duration: int = WINDOW_SEC) -> np.ndarray:
    """Generates synthetic PPG matching given HR, SpO2, and stress level."""
    T = fs * duration
    t = np.linspace(0, duration, T)
    rr = fs * 60 / hr
    # HRV jitter based on stress (high stress = low HRV = regular intervals)
    jitter_std = {'Low': 0.05, 'Medium': 0.02, 'High': 0.005}[stress]
    ppg = np.zeros(T)
    beat_pos = 0
    while beat_pos < T:
        jitter = int(np.random.normal(0, jitter_std * fs))
        pos    = int(beat_pos) + jitter
        width  = max(8, int(fs * 0.3))
        if 0 < pos < T - width:
            ppg[pos:pos+width] += scipy_signal.windows.gaussian(width, width//4)
        beat_pos += rr
    # SpO2 amplitude effect: lower SpO2 → reduced AC amplitude
    amplitude = (spo2 - 85) / 15.0
    ppg *= amplitude
    ppg += np.random.normal(0, 0.03, T)
    return ppg_prep(ppg)


print('Signal generators ready ✓')

In [ ]:
class WearableDataset(Dataset):
    """
    Synthetic wearable dataset.
    Replace generate_* calls with real data loading when available.
    """
    def __init__(self, n_samples: int = 2000):
        self.samples = []
        rhythm_weights = [0.45, 0.2, 0.1, 0.15, 0.1]   # class balance

        for _ in range(n_samples):
            rhythm = np.random.choice(RHYTHM_CLASSES, p=rhythm_weights)
            stress = np.random.choice(STRESS_CLASSES, p=[0.5, 0.35, 0.15])

            # HR varies by rhythm
            hr_map = {
                'Normal':      np.random.uniform(60, 100),
                'AFib':        np.random.uniform(60, 130),
                'Bradycardia': np.random.uniform(HR_CRITICAL_LOW, HR_BRADY_THRESH),
                'Tachycardia': np.random.uniform(HR_TACHY_THRESH, HR_CRITICAL_HIGH),
                'Anomaly':     np.random.uniform(50, 130)
            }
            hr = hr_map[rhythm]

            # SpO2 varies by severity
            if rhythm in ['Bradycardia', 'Anomaly'] and np.random.rand() < 0.3:
                spo2 = np.random.uniform(85, SPO2_WARN_THRESH)
            else:
                spo2 = np.random.uniform(95, 100)

            ecg = generate_ecg(rhythm)
            ppg = generate_ppg(hr, spo2, stress)

            # Compute severity label (ground truth for training)
            severity = self._compute_severity(rhythm, hr, spo2, stress)

            self.samples.append({
                'ecg':           ecg,
                'ppg':           ppg,
                'rhythm_label':  RHYTHM_CLASSES.index(rhythm),
                'stress_label':  STRESS_CLASSES.index(stress),
                'spo2':          spo2,
                'severity':      severity
            })

    @staticmethod
    def _compute_severity(rhythm, hr, spo2, stress) -> float:
        score = 0.0
        if rhythm == 'AFib':        score += 0.45
        if rhythm == 'Anomaly':     score += 0.35
        if rhythm == 'Tachycardia': score += 0.30
        if rhythm == 'Bradycardia': score += 0.30
        if hr < HR_CRITICAL_LOW:    score += 0.40
        if hr > HR_CRITICAL_HIGH:   score += 0.35
        if spo2 < SPO2_CRIT_THRESH: score += 0.40
        elif spo2 < SPO2_WARN_THRESH: score += 0.20
        if stress == 'High':        score += 0.15
        elif stress == 'Medium':    score += 0.05
        return float(np.clip(score, 0.0, 1.0))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return {
            'ecg':          torch.FloatTensor(s['ecg']).unsqueeze(0),
            'ppg':          torch.FloatTensor(s['ppg']).unsqueeze(0),
            'rhythm_label': torch.LongTensor([s['rhythm_label']]).squeeze(),
            'stress_label': torch.LongTensor([s['stress_label']]).squeeze(),
            'spo2':         torch.FloatTensor([s['spo2']]),
            'severity':     torch.FloatTensor([s['severity']])
        }


# Choose the training source.
# 'auto' uses PTB-XL real ECG if wfdb is installed and files are present;
# otherwise it falls back to the synthetic wearable dataset.
DATA_SOURCE = 'auto'   # options: 'auto', 'ptbxl', 'synthetic'
PTBXL_MAX_RECORDS = 2000

print('Building dataset...', end=' ')
use_ptbxl = (
    DATA_SOURCE in {'auto', 'ptbxl'} and
    'PTBXLWearableDataset' in globals() and
    globals().get('WFDB_AVAILABLE', False) and
    PTBXL_DATABASE.exists() and
    SCP_STATEMENTS.exists()
)

if use_ptbxl:
    full_ds = PTBXLWearableDataset(max_records=PTBXL_MAX_RECORDS, sampling_rate=100)
    dataset_name = 'PTB-XL real ECG'
elif DATA_SOURCE == 'ptbxl':
    raise RuntimeError('PTB-XL requested, but wfdb or PTB-XL files are unavailable.')
else:
    full_ds = WearableDataset(n_samples=2000)
    dataset_name = 'synthetic ECG+PPG'

train_size = int(0.8 * len(full_ds))
val_size   = len(full_ds) - train_size
split_gen = torch.Generator().manual_seed(42)
train_ds, val_ds = torch.utils.data.random_split(full_ds, [train_size, val_size], generator=split_gen)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'done ? {dataset_name} | train: {train_size}  val: {val_size}')


## 13 · Training — Multi-Task Loss

In [ ]:
def multi_task_loss(outputs: dict, batch: dict, device: str) -> Tuple[torch.Tensor, dict]:
    """
    Combined loss across all four prediction heads.
    Each task is weighted by clinical importance.
    """
    rhythm_loss   = F.cross_entropy(
        outputs['rhythm_logits'], batch['rhythm_label'].to(device)
    )
    stress_loss   = F.cross_entropy(
        outputs['stress_logits'], batch['stress_label'].to(device)
    )
    spo2_loss     = F.mse_loss(
        outputs['spo2_pred'].squeeze(), batch['spo2'].squeeze().to(device)
    ) / 100.0     # normalise % scale
    severity_loss = F.binary_cross_entropy(
        outputs['severity_score'].squeeze(), batch['severity'].squeeze().to(device)
    )

    # Weights reflect clinical priority
    total = (1.5 * rhythm_loss +
             1.0 * severity_loss +
             0.8 * stress_loss +
             0.5 * spo2_loss)

    return total, {
        'rhythm':   rhythm_loss.item(),
        'stress':   stress_loss.item(),
        'spo2':     spo2_loss.item(),
        'severity': severity_loss.item()
    }


optimizer = AdamW(triage_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = OneCycleLR(optimizer, max_lr=LR,
                       steps_per_epoch=len(train_loader), epochs=EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'train_rhythm_acc': [], 'val_rhythm_acc': []}
best_val, CKPT = float('inf'), 'triage_best.pt'

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    triage_model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    for batch in train_loader:
        ecg = batch['ecg'].to(DEVICE)
        ppg = batch['ppg'].to(DEVICE)
        outputs = triage_model(ecg, ppg)
        loss, _ = multi_task_loss(outputs, batch, DEVICE)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(triage_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        tr_loss += loss.item() * ecg.size(0)
        preds = outputs['rhythm_logits'].argmax(dim=1)
        tr_correct += (preds == batch['rhythm_label'].to(DEVICE)).sum().item()
        tr_total   += ecg.size(0)

    # ── Validate ──
    triage_model.eval()
    vl_loss, vl_correct, vl_total = 0.0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            ecg = batch['ecg'].to(DEVICE)
            ppg = batch['ppg'].to(DEVICE)
            outputs = triage_model(ecg, ppg)
            loss, _ = multi_task_loss(outputs, batch, DEVICE)
            vl_loss += loss.item() * ecg.size(0)
            preds = outputs['rhythm_logits'].argmax(dim=1)
            vl_correct += (preds == batch['rhythm_label'].to(DEVICE)).sum().item()
            vl_total   += ecg.size(0)

    tr_loss /= tr_total; vl_loss /= vl_total
    tr_acc   = tr_correct / tr_total
    vl_acc   = vl_correct / vl_total
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_rhythm_acc'].append(tr_acc)
    history['val_rhythm_acc'].append(vl_acc)

    if vl_loss < best_val:
        best_val = vl_loss
        torch.save(triage_model.state_dict(), CKPT)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS}  Loss: {tr_loss:.4f}/{vl_loss:.4f}  '
              f'Rhythm Acc: {tr_acc:.3f}/{vl_acc:.3f}')

triage_model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
print(f'\nBest val loss: {best_val:.4f}')

## 14 · Severity Scoring Engine

In [ ]:
class SeverityEngine:
    """
    Combines model predictions with rule-based physiological checks
    to produce a final severity tier.

    Rule-based overrides exist because the model should NEVER miss
    a critically dangerous vital sign, even if it hasn't been trained on it.
    """
    def __init__(self, model: nn.Module, device: str = DEVICE):
        self.model  = model
        self.device = device
        self.model.eval()

    def _classify_severity(self, score: float, vitals: dict,
                            rhythm: str) -> Tuple[Severity, str]:
        """Determines tier from score + hard physiological rules."""
        reasons = []

        # ── Hard RED rules (physiological emergency overrides) ────────────
        if vitals['heart_rate'] < HR_CRITICAL_LOW:
            return Severity.RED, f'Critical bradycardia: {vitals["heart_rate"]:.0f} bpm'
        if vitals['heart_rate'] > HR_CRITICAL_HIGH:
            return Severity.RED, f'Critical tachycardia: {vitals["heart_rate"]:.0f} bpm'
        if vitals['spo2'] < SPO2_CRIT_THRESH:
            return Severity.RED, f'Critical hypoxia: SpO2 {vitals["spo2"]:.1f}%'
        if rhythm == 'AFib' and score > 0.6:
            return Severity.RED, 'High-confidence AFib detected'

        # ── Model score tiers ─────────────────────────────────────────────
        if score >= RED_THRESH:
            reasons.append(f'Severity score {score:.2f}')
            if vitals['spo2'] < SPO2_WARN_THRESH:
                reasons.append(f'Low SpO2 {vitals["spo2"]:.1f}%')
            return Severity.RED, ' + '.join(reasons)

        if score >= YELLOW_THRESH:
            if vitals['spo2'] < SPO2_WARN_THRESH:
                reasons.append(f'SpO2 {vitals["spo2"]:.1f}%')
            if rhythm != 'Normal':
                reasons.append(f'{rhythm} detected')
            if vitals['hrv_rmssd'] < HRV_LOW_THRESH:
                reasons.append(f'Low HRV {vitals["hrv_rmssd"]:.1f}ms')
            reason = ' + '.join(reasons) if reasons else f'Score {score:.2f}'
            return Severity.YELLOW, reason

        return Severity.GREEN, ''

    @torch.no_grad()
    def evaluate(self, ecg_raw: np.ndarray, ppg_raw: np.ndarray,
                 gps: Optional[Tuple[float, float]] = None) -> TriageResult:
        # Preprocess
        ecg = ecg_prep(ecg_raw)
        ppg = ppg_prep(ppg_raw)

        # Extract rule-based vitals
        vitals = feat_ext.extract_all(ecg, ppg)

        # Model inference
        ecg_t = torch.FloatTensor(ecg).unsqueeze(0).unsqueeze(0).to(self.device)
        ppg_t = torch.FloatTensor(ppg).unsqueeze(0).unsqueeze(0).to(self.device)
        outputs = self.model(ecg_t, ppg_t)

        rhythm_probs   = F.softmax(outputs['rhythm_logits'], dim=1).cpu().numpy()[0]
        stress_probs   = F.softmax(outputs['stress_logits'], dim=1).cpu().numpy()[0]
        spo2_pred      = outputs['spo2_pred'].cpu().item()
        severity_score = outputs['severity_score'].cpu().item()

        rhythm_idx   = int(rhythm_probs.argmax())
        rhythm_label = RHYTHM_CLASSES[rhythm_idx]
        stress_label = STRESS_CLASSES[int(stress_probs.argmax())]

        # Override vitals SpO2 with model estimate (blend both sources)
        blended_spo2 = 0.6 * spo2_pred + 0.4 * vitals['spo2']
        vitals['spo2'] = blended_spo2

        # Severity classification
        severity, reason = self._classify_severity(severity_score, vitals, rhythm_label)

        return TriageResult(
            timestamp        = datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            severity         = severity,
            severity_score   = severity_score,
            rhythm_label     = rhythm_label,
            rhythm_probs     = {c: float(f'{p:.4f}') for c, p in
                                zip(RHYTHM_CLASSES, rhythm_probs)},
            heart_rate       = vitals['heart_rate'],
            hrv_rmssd        = vitals['hrv_rmssd'],
            spo2             = blended_spo2,
            stress_level     = stress_label,
            stress_probs     = {c: float(f'{p:.4f}') for c, p in
                                zip(STRESS_CLASSES, stress_probs)},
            requires_escalation = (severity == Severity.RED),
            escalation_reason   = reason,
            gps_location        = gps
        )


engine = SeverityEngine(triage_model)
print('Severity engine ready ✓')

## 15 · Emergency Escalation System

In [ ]:
@dataclass
class UserProfile:
    name:              str
    age:               int
    blood_type:        str
    known_conditions:  List[str]
    medications:       List[str]
    emergency_contact: Dict[str, str]   # {'name': ..., 'phone': ..., 'relation': ...}
    emergency_number:  str = '911'      # localised per country


class EscalationSystem:
    """
    Handles emergency escalation when severity = RED.

    In production this connects to:
    - Twilio / Firebase for SMS + push notifications
    - Platform emergency SOS APIs (iOS CallKit / Android TelephonyManager)
    - Optionally streams last 60s of ECG to emergency services

    The mock methods here log what would be sent in a real deployment.
    """
    def __init__(self, user: UserProfile):
        self.user = user
        self.log  = []

    def _format_medical_summary(self, result: TriageResult) -> str:
        loc_str = (f'{result.gps_location[0]:.5f},{result.gps_location[1]:.5f}'
                   if result.gps_location else 'unavailable')
        return (
            f"CARDIAC ALERT — HelixMind Health Monitor\n"
            f"Patient : {self.user.name}, Age {self.user.age}, "
            f"Blood type {self.user.blood_type}\n"
            f"Time    : {result.timestamp}\n"
            f"GPS     : {loc_str}\n"
            f"\nVITALS:\n"
            f"  Heart rate : {result.heart_rate:.0f} bpm\n"
            f"  SpO2       : {result.spo2:.1f}%\n"
            f"  Rhythm     : {result.rhythm_label} "
            f"({result.rhythm_probs[result.rhythm_label]:.1%} confidence)\n"
            f"  Stress     : {result.stress_level}\n"
            f"  Severity   : {result.severity_score:.2f}/1.00\n"
            f"\nALERT REASON: {result.escalation_reason}\n"
            f"\nMEDICAL HISTORY:\n"
            f"  Conditions : {', '.join(self.user.known_conditions) or 'None'}\n"
            f"  Medications: {', '.join(self.user.medications) or 'None'}"
        )

    def notify_emergency_contact(self, result: TriageResult):
        """Sends SMS + push notification to pre-set emergency contact."""
        contact = self.user.emergency_contact
        message = (
            f"🔴 HEALTH ALERT — {self.user.name} may need help.\n"
            f"Detected: {result.escalation_reason}\n"
            f"HR: {result.heart_rate:.0f} bpm | SpO2: {result.spo2:.1f}%\n"
            + (f"Last known location: https://maps.google.com/?"
               f"q={result.gps_location[0]},{result.gps_location[1]}"
               if result.gps_location else 'Location unavailable')
        )
        # ── In production: Twilio SMS / Firebase push notification ────────
        entry = {
            'action':    'NOTIFY_CONTACT',
            'to':        f"{contact['name']} ({contact['relation']})",
            'phone':     contact['phone'],
            'message':   message,
            'timestamp': result.timestamp
        }
        self.log.append(entry)
        print(f'[CONTACT ALERT → {contact["name"]}]')
        print(message)

    def contact_emergency_services(self, result: TriageResult):
        """Calls emergency services with full medical briefing."""
        medical_brief = self._format_medical_summary(result)
        # ── In production: platform SOS API (iOS / Android) ───────────────
        # iOS  : CXCallController to dial + share HealthKit summary
        # Android: TelephonyManager + SMS to emergency dispatch
        entry = {
            'action':       'EMERGENCY_SERVICES',
            'number':       self.user.emergency_number,
            'medical_brief': medical_brief,
            'timestamp':    result.timestamp
        }
        self.log.append(entry)
        print(f'\n[EMERGENCY SERVICES → {self.user.emergency_number}]')
        print(medical_brief)

    def escalate(self, result: TriageResult):
        """Triggers full simultaneous escalation for RED severity."""
        if not result.requires_escalation:
            return
        print('=' * 60)
        print('🔴  RED ALERT — INITIATING EMERGENCY ESCALATION')
        print('=' * 60)
        self.notify_emergency_contact(result)
        print()
        self.contact_emergency_services(result)
        print('=' * 60)
        print(f'Escalation complete. {len(self.log)} action(s) logged.')


print('Escalation system defined ✓')

## 16 · End-to-End Demo

In [ ]:
# ── Set up a sample user profile ─────────────────────────────────────────────
user = UserProfile(
    name              = 'John Doe',
    age               = 45,
    blood_type        = 'O+',
    known_conditions  = ['Hypertension', 'Type 2 Diabetes'],
    medications       = ['Metformin 500mg', 'Lisinopril 10mg'],
    emergency_contact = {'name': 'Jane Doe', 'phone': '+1-555-0100', 'relation': 'Spouse'},
    emergency_number  = '911'
)

escalation_system = EscalationSystem(user)

# ── Simulate three scenarios ──────────────────────────────────────────────────
scenarios = [
    ('Normal',      'Low',    97.0, (4.65, 7.45)),    # GPS: sample coords
    ('AFib',        'High',   94.0, (4.65, 7.45)),
    ('Bradycardia', 'Medium', 88.0, (4.65, 7.45)),    # Critical hypoxia
]

print('\n' + '='*60)
for rhythm, stress, spo2, gps in scenarios:
    hr = {'Normal': 75, 'AFib': 110, 'Bradycardia': 38}[rhythm]
    ecg_raw = generate_ecg(rhythm)
    ppg_raw = generate_ppg(hr, spo2, stress)

    result = engine.evaluate(ecg_raw, ppg_raw, gps=gps)
    print(result.summary())

    if result.requires_escalation:
        escalation_system.escalate(result)

    print('='*60)

## 17 · Continuous Monitoring Loop (Production Pattern)

In [ ]:
def continuous_monitor(engine: SeverityEngine,
                        escalation: EscalationSystem,
                        get_ecg_fn,      # callable → np.ndarray from device SDK
                        get_ppg_fn,      # callable → np.ndarray from device SDK
                        get_gps_fn,      # callable → (lat, lon) or None
                        interval_sec: int = 30,
                        max_iters: int = None):
    """
    Production monitoring loop.

    Runs inference every `interval_sec` seconds.
    Plug in real wearable SDK callbacks for get_ecg_fn, get_ppg_fn, get_gps_fn.

    Apple Watch:  HealthKit HKElectrocardiogram + HKQuantityTypeIdentifierHeartRate
    Android/WearOS: Health Services API (ExerciseClient, PassiveMonitoringClient)
    KardiaMobile: AliveCor SDK
    """
    print('HelixMind continuous monitoring started...')
    history = []
    i = 0
    while max_iters is None or i < max_iters:
        ecg = get_ecg_fn()
        ppg = get_ppg_fn()
        gps = get_gps_fn()

        result = engine.evaluate(ecg, ppg, gps)
        history.append(result)
        print(result.summary())

        if result.requires_escalation:
            escalation.escalate(result)
            break    # in production: continue monitoring but flag as escalated

        i += 1
        if max_iters is None:
            time.sleep(interval_sec)

    return history


# ── Simulate 5 monitoring windows ────────────────────────────────────────────
def mock_ecg(): return generate_ecg(np.random.choice(['Normal','Normal','AFib']))
def mock_ppg(): return generate_ppg(np.random.uniform(60, 110), np.random.uniform(94, 100), 'Low')
def mock_gps(): return (4.6500, 7.4500)

monitor_log = continuous_monitor(
    engine, escalation_system,
    get_ecg_fn = mock_ecg,
    get_ppg_fn = mock_ppg,
    get_gps_fn = mock_gps,
    max_iters  = 5
)

## 18 · Export for Mobile Deployment

In [ ]:
# ── Save PyTorch checkpoint ───────────────────────────────────────────────────
torch.save({
    'model_state_dict':  triage_model.state_dict(),
    'ecg_fs':            ECG_FS,
    'ppg_fs':            PPG_FS,
    'window_sec':        WINDOW_SEC,
    'rhythm_classes':    RHYTHM_CLASSES,
    'stress_classes':    STRESS_CLASSES,
    'yellow_thresh':     YELLOW_THRESH,
    'red_thresh':        RED_THRESH,
}, './helixmind_triage.pt')
print('Checkpoint saved ✓')

# ── Export to ONNX → TFLite (Android) / CoreML (iOS) ────────────────────────
triage_model.eval()
dummy_ecg = torch.randn(1, 1, ECG_LEN)
dummy_ppg = torch.randn(1, 1, PPG_LEN)

# ONNX export with named inputs
torch.onnx.export(
    triage_model,
    (dummy_ecg, dummy_ppg),
    './helixmind_triage.onnx',
    opset_version = 13,
    input_names   = ['ecg', 'ppg'],
    output_names  = ['rhythm_logits', 'stress_logits', 'spo2_pred', 'severity_score'],
    dynamic_axes  = {
        'ecg':            {0: 'batch'},
        'ppg':            {0: 'batch'},
        'rhythm_logits':  {0: 'batch'},
        'severity_score': {0: 'batch'}
    }
)
print('ONNX export → helixmind_triage.onnx ✓')
print('\nNext steps for mobile deployment:')
print('  Android : onnx → onnxruntime-android or tflite via onnx2tf')
print('  iOS     : onnx → CoreML via coremltools.converters.onnx')
print('  Both    : quantise to INT8 for ~4× size reduction')

---
## System Architecture Summary

```
WEARABLE DEVICE (Apple Watch / KardiaMobile / Fitbit / Garmin)
         │
         ├─ Single-lead ECG  (256 Hz, 30s window → 7680 samples)
         └─ PPG signal       ( 64 Hz, 30s window → 1920 samples)
                  │
         ┌────────▼────────┐
         │  Preprocessing  │  Bandpass filter · Z-score norm · artefact clip
         └────────┬────────┘
                  │
         ┌────────▼──────────────────────────────────┐
         │         WearableTriageModel (<500K)         │
         │                                            │
         │  ECG branch ──┐                            │
         │  (DW-Sep CNN) ├─ Fusion Conv ─ BiLSTM ─┐  │
         │  PPG branch ──┘                         │  │
         │  (DW-Sep CNN)                           │  │
         │                              ┌──────────┘  │
         │                     Rhythm · Stress · SpO2 · Severity│
         └────────────────────────────────────────────┘
                  │
         ┌────────▼────────┐
         │  SeverityEngine  │  Model score + physiological hard rules
         └────────┬────────┘
                  │
        ┌─────────┼──────────┐
        ▼         ▼          ▼
      GREEN     YELLOW      RED
    Continue   User alert  ┌──────────────────────────┐
    monitor    + recommend  │  EscalationSystem        │
               clinical ECG│  ├─ SMS → Emergency contact│
                           │  └─ Call → 911 + GPS + ECG│
                           └──────────────────────────┘
                                        │
                            ┌───────────▼────────────┐
                            │   TIER 2 HANDOFF       │
                            │  12-lead CNN + BiLSTM  │
                            │  (Clinical notebook)   │
                            └────────────────────────┘
```

### Severity tier logic
| Tier | Score | Hard rule overrides |
|---|---|---|
| 🟢 GREEN | < 0.40 | — |
| 🟡 YELLOW | 0.40–0.72 | SpO2 < 95% · abnormal rhythm · low HRV |
| 🔴 RED | > 0.72 | HR < 40 or > 150 · SpO2 < 90% · high-conf AFib |

### Why depthwise separable convolutions
Standard Conv1d(128, 128, k=7) = 128 × 128 × 7 = **114,688** params  
Depthwise separable equivalent = (128×7) + (128×128) = **17,280** params — **6.6× fewer**

This is what keeps the model under 500K params while retaining the same receptive field.